# Fruit Image Classification — Complete AI Assignment

**Dataset:** Kaggle `moltean/fruits` (Fruits-360)  
**Models:** Custom CNN, MobileNetV2, ResNet50  
**Output:** Accuracy, Precision, Recall, F1, inference speed, saved best model for Streamlit.

This notebook follows the assignment workflow: dataset acquisition → preprocessing/augmentation → three algorithms → evaluation/comparison → model export. Exact results are generated only after training.

In [ ]:
# 1. Install packages (run once)
%pip install -q --upgrade kagglehub tensorflow scikit-learn pandas matplotlib Pillow

In [ ]:
# 2. Imports and configuration
from pathlib import Path
import json, time, numpy as np, pandas as pd, matplotlib.pyplot as plt
import kagglehub, tensorflow as tf
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
SEED=123
tf.keras.utils.set_random_seed(SEED)
IMG_SIZE=(128,128); BATCH_SIZE=32; VAL_SPLIT=0.20; AUTOTUNE=tf.data.AUTOTUNE
MODEL_DIR=Path('./fruits360_full_project/models'); OUTPUT_DIR=Path('./fruits360_full_project/outputs')
MODEL_DIR.mkdir(parents=True,exist_ok=True); OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
print('TensorFlow:',tf.__version__); print('GPU:',tf.config.list_physical_devices('GPU'))

## 3. Dataset and preprocessing
The notebook downloads Fruits-360 directly with KaggleHub, prefers the standardized **100×100** Training/Test branch when available, discovers the class count automatically, uses an 80/20 train-validation split, resizes to 128×128 RGB, applies augmentation, and uses integer label encoding from directory names.

In [ ]:
# 3.1 Download dataset and find Training/Test
DATA_ROOT=Path('./data/fruits360_kaggle'); DATA_ROOT.mkdir(parents=True,exist_ok=True)
try:
    dataset_path=Path(kagglehub.dataset_download('moltean/fruits',output_dir=str(DATA_ROOT)))
except Exception:
    kagglehub.login()
    dataset_path=Path(kagglehub.dataset_download('moltean/fruits',output_dir=str(DATA_ROOT)))

def class_count(p):
    try: return sum(x.is_dir() for x in p.iterdir())
    except Exception: return 0

def find_split(root,name):
    c=[p for p in root.rglob(name) if p.is_dir()]
    if not c: raise FileNotFoundError(f'No {name} folder found')
    return sorted(c,key=lambda p:('100x100' in str(p).lower(),class_count(p)),reverse=True)[0]

TRAIN_DIR=find_split(dataset_path,'Training'); TEST_DIR=find_split(dataset_path,'Test')
print('Dataset:',dataset_path.resolve()); print('Training:',TRAIN_DIR); print('Test:',TEST_DIR); print('Classes:',class_count(TRAIN_DIR))

In [ ]:
# 3.2 TensorFlow datasets + augmentation
train_ds=tf.keras.utils.image_dataset_from_directory(TRAIN_DIR,validation_split=VAL_SPLIT,subset='training',seed=SEED,image_size=IMG_SIZE,batch_size=BATCH_SIZE,label_mode='int',shuffle=True)
val_ds=tf.keras.utils.image_dataset_from_directory(TRAIN_DIR,validation_split=VAL_SPLIT,subset='validation',seed=SEED,image_size=IMG_SIZE,batch_size=BATCH_SIZE,label_mode='int',shuffle=False)
test_ds=tf.keras.utils.image_dataset_from_directory(TEST_DIR,image_size=IMG_SIZE,batch_size=BATCH_SIZE,label_mode='int',shuffle=False)
CLASS_NAMES=train_ds.class_names; NUM_CLASSES=len(CLASS_NAMES)
train_ds=train_ds.prefetch(AUTOTUNE); val_ds=val_ds.prefetch(AUTOTUNE); test_ds=test_ds.prefetch(AUTOTUNE)
augmentation=tf.keras.Sequential([tf.keras.layers.RandomFlip('horizontal'),tf.keras.layers.RandomRotation(.12),tf.keras.layers.RandomZoom(.10),tf.keras.layers.RandomContrast(.10)])
print('Number of classes:',NUM_CLASSES); print(CLASS_NAMES[:30])

## 4. Algorithms
**Custom CNN** is the baseline trained from scratch. **MobileNetV2** and **ResNet50** use ImageNet transfer learning: first freeze the pretrained base, then fine-tune upper layers with a low learning rate. MobileNetV2 is deployment-efficient; ResNet50 has higher capacity but is larger/slower.

In [ ]:
# 4.1 Model builders and training settings
QUICK_RUN=False  # True only for a short code test
CNN_EPOCHS=2 if QUICK_RUN else 8; TRANSFER_EPOCHS=2 if QUICK_RUN else 8; FINETUNE_EPOCHS=1 if QUICK_RUN else 4
callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=2,restore_best_weights=True),tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss',factor=.3,patience=1,min_lr=1e-7)]

def build_cnn():
    i=tf.keras.Input(shape=IMG_SIZE+(3,)); x=augmentation(i); x=tf.keras.layers.Rescaling(1/255.)(x)
    for f in (32,64,128):
        x=tf.keras.layers.Conv2D(f,3,padding='same',activation='relu')(x); x=tf.keras.layers.BatchNormalization()(x); x=tf.keras.layers.MaxPooling2D()(x)
    x=tf.keras.layers.Conv2D(192,3,padding='same',activation='relu')(x); x=tf.keras.layers.GlobalAveragePooling2D()(x); x=tf.keras.layers.Dropout(.30)(x); o=tf.keras.layers.Dense(NUM_CLASSES)(x)
    m=tf.keras.Model(i,o,name='Custom_CNN'); m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),metrics=['accuracy']); return m

def build_transfer(name):
    if name=='MobileNetV2': base=tf.keras.applications.MobileNetV2(include_top=False,weights='imagenet',input_shape=IMG_SIZE+(3,)); prep=tf.keras.applications.mobilenet_v2.preprocess_input
    else: base=tf.keras.applications.ResNet50(include_top=False,weights='imagenet',input_shape=IMG_SIZE+(3,)); prep=tf.keras.applications.resnet50.preprocess_input
    base.trainable=False; i=tf.keras.Input(shape=IMG_SIZE+(3,)); x=augmentation(i); x=prep(x); x=base(x,training=False); x=tf.keras.layers.GlobalAveragePooling2D()(x); x=tf.keras.layers.Dropout(.25)(x); o=tf.keras.layers.Dense(NUM_CLASSES)(x)
    m=tf.keras.Model(i,o,name=name); m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),metrics=['accuracy']); return m,base

In [ ]:
# 4.2 Train all three algorithms
custom_cnn=build_cnn(); h_cnn=custom_cnn.fit(train_ds,validation_data=val_ds,epochs=CNN_EPOCHS,callbacks=callbacks); custom_cnn.save(MODEL_DIR/'custom_cnn.keras')

mobilenet,mobile_base=build_transfer('MobileNetV2'); h_m1=mobilenet.fit(train_ds,validation_data=val_ds,epochs=TRANSFER_EPOCHS,callbacks=callbacks); mobile_base.trainable=True
for l in mobile_base.layers[:-30]: l.trainable=False
mobilenet.compile(optimizer=tf.keras.optimizers.Adam(1e-5),loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),metrics=['accuracy']); h_m2=mobilenet.fit(train_ds,validation_data=val_ds,epochs=FINETUNE_EPOCHS,callbacks=callbacks); mobilenet.save(MODEL_DIR/'mobilenetv2.keras')

resnet,res_base=build_transfer('ResNet50'); h_r1=resnet.fit(train_ds,validation_data=val_ds,epochs=TRANSFER_EPOCHS,callbacks=callbacks); res_base.trainable=True
for l in res_base.layers[:-25]: l.trainable=False
resnet.compile(optimizer=tf.keras.optimizers.Adam(1e-5),loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),metrics=['accuracy']); h_r2=resnet.fit(train_ds,validation_data=val_ds,epochs=FINETUNE_EPOCHS,callbacks=callbacks); resnet.save(MODEL_DIR/'resnet50.keras')

## 5. Evaluation
The independent Test folder is used for final **Accuracy**, **Macro Precision**, **Macro Recall**, **Macro F1**, and inference speed. Macro averaging gives each class equal importance.

In [ ]:
# 5.1 Evaluate and compare
def evaluate(model,name):
    yt=[]; yp=[]; start=time.perf_counter()
    for x,y in test_ds:
        pred=np.argmax(model.predict(x,verbose=0),axis=1); yt.extend(y.numpy()); yp.extend(pred)
    sec=time.perf_counter()-start; acc=accuracy_score(yt,yp); p,r,f,_=precision_recall_fscore_support(yt,yp,average='macro',zero_division=0)
    pd.DataFrame(classification_report(yt,yp,target_names=CLASS_NAMES,output_dict=True,zero_division=0)).T.to_csv(OUTPUT_DIR/f'{name}_classification_report.csv')
    np.save(OUTPUT_DIR/f'{name}_confusion_matrix.npy',confusion_matrix(yt,yp))
    return {'Model':name,'Accuracy':acc,'Macro Precision':p,'Macro Recall':r,'Macro F1':f,'Inference seconds':sec,'Images / second':len(yt)/sec}

results=pd.DataFrame([evaluate(custom_cnn,'Custom_CNN'),evaluate(mobilenet,'MobileNetV2'),evaluate(resnet,'ResNet50')]).sort_values('Macro F1',ascending=False)
display(results); results.to_csv(OUTPUT_DIR/'model_comparison.csv',index=False)
ax=results.set_index('Model')[['Accuracy','Macro Precision','Macro Recall','Macro F1']].plot(kind='bar',figsize=(10,5)); ax.set_ylim(0,1.05); ax.set_ylabel('Score'); plt.xticks(rotation=0); plt.tight_layout(); plt.savefig(OUTPUT_DIR/'model_comparison.png',dpi=180); plt.show()

In [ ]:
# 5.2 Save best model, labels and metadata for Streamlit
best=results.iloc[0]; models={'Custom_CNN':custom_cnn,'MobileNetV2':mobilenet,'ResNet50':resnet}; best_model=models[best['Model']]
best_model.save(MODEL_DIR/'best_fruit_model.keras')
(MODEL_DIR/'class_names.json').write_text(json.dumps(CLASS_NAMES,indent=2,ensure_ascii=False),encoding='utf-8')
meta={'best_model':best['Model'],'image_size':list(IMG_SIZE),'num_classes':NUM_CLASSES,'test_accuracy':float(best['Accuracy']),'macro_precision':float(best['Macro Precision']),'macro_recall':float(best['Macro Recall']),'macro_f1':float(best['Macro F1'])}
(MODEL_DIR/'model_metadata.json').write_text(json.dumps(meta,indent=2),encoding='utf-8')
print(meta); print('Streamlit files saved in:',MODEL_DIR.resolve())

## 6. Performance discussion and conclusion
Use `outputs/model_comparison.csv` in the report. Discuss: which model has the best test accuracy/F1; whether the gain justifies larger inference cost; why MobileNetV2 may be preferred for live deployment; and why Fruits-360's controlled backgrounds can produce better dataset accuracy than cluttered real-world webcam scenes.

**Advantages/Disadvantages:** Custom CNN is easy to explain but starts from random weights. MobileNetV2 is efficient and pretrained but lower capacity. ResNet50 has strong representation capacity but larger memory and slower inference. A future extension can use YOLO for multi-object bounding-box detection.

### References
- Kaggle Fruits-360 (`moltean/fruits`)
- KaggleHub official Python library
- TensorFlow transfer learning/fine-tuning documentation
- TensorFlow `image_dataset_from_directory` documentation
- Muresan & Oltean (2018), *Fruit recognition from images using deep learning*.